# Kart Engine Audio Analysis - Sodi GT5 (Honda GX390), in-helmet AirPods

Pull driving telemetry out of an in-helmet AirPods recording of a Sodi GT5 rental run.

**Tuned for this setup**
- **Engine:** Honda GX390 - 4-stroke single, fires once per 2 revs, so `firing_Hz = RPM / 120`.
- **Rev range:** governed rental engine, roughly 1500 rpm idle to a ~4000-4500 rpm limiter -> firing fundamental only **~12-40 Hz**. That's why the tracking band sits so low.
- **In-helmet mic:** the shell low-passes the airborne note and structure-borne vibration reinforces the low end, so the engine energy lives ~15-300 Hz. The AirPod's voice high-pass may sap the fundamental itself - harmonic summation recovers f0 from the harmonic ladder anyway.

**Before you run**
- `ffmpeg` installed: `brew install ffmpeg`
- deps in the install cell below
- Expect a **compressed RPM swing** (governed engine) and a quiet exhaust - the loudness/centroid proxies in section 5 carry a lot of the signal.

In [ ]:
# Run once if needed. ffmpeg is a SYSTEM package:
#   macOS:  brew install ffmpeg
# !pip install librosa soundfile scipy numpy matplotlib pandas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa, librosa.display
from scipy.signal import medfilt
from scipy.ndimage import uniform_filter1d

plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["figure.dpi"] = 110

## 1. Config

Defaults are set for the GT5/GX390 in-helmet case. After you've seen the spectrogram in section 3, nudge `FMIN/FMAX` to bracket the firing fundamental (the lowest bright moving band).

In [ ]:
# ---- EDIT THESE ----
AUDIO_PATH = "kart_run.mp4"

# Honda GX390 = 4-stroke single. Leave these unless you swap karts.
ENGINE_STROKE = 4
CYLINDERS     = 1

# Firing-fundamental search band (Hz). GX390 idle->limiter spans ~12-40 Hz.
# Keep FMIN low so the true fundamental is a candidate even if the AirPod
# attenuated it - harmonic summation scores it from 2f, 3f, 4f...
FMIN = 11.0
FMAX = 48.0

# If max RPM lands ~2x/3x what the GX390 can physically turn (~4500 rpm cap),
# the tracker latched a harmonic -> set this to 2 or 3 to rescale.
HARMONIC_TRACKED = 1

# Low frequencies need fine FFT resolution, so we trade sample rate for it.
SR     = 11025          # plenty: we only care about <300 Hz
N_FFT  = 8192           # ~1.35 Hz bins -> ~160 rpm resolution
HOP    = 256            # ~43 frames/sec
N_HARM = 8              # harmonics summed per f0 candidate (covers ~12-380 Hz)
# --------------------

FIRINGS_PER_REV = CYLINDERS * (1.0 if ENGINE_STROKE == 2 else 0.5)
RPM_PER_HZ = (60.0 / FIRINGS_PER_REV) / HARMONIC_TRACKED
print(f"RPM = firing_Hz * {RPM_PER_HZ:.0f}   (GX390 sanity: ~1500 idle, ~4000-4500 at the limiter)")

## 2. Load audio

librosa first; ffmpeg fallback for the m4a/mp4 codec.

In [ ]:
def load_audio(path, target_sr=SR):
    try:
        return librosa.load(path, sr=target_sr, mono=True)
    except Exception as e:
        print(f"librosa load failed ({e}); using ffmpeg")
        import subprocess, tempfile, os
        tmp = tempfile.mktemp(suffix=".wav")
        subprocess.run(["ffmpeg", "-y", "-i", path, "-ac", "1", "-ar", str(target_sr), tmp],
                       check=True, capture_output=True)
        y, sr = librosa.load(tmp, sr=target_sr, mono=True)
        os.remove(tmp)
        return y, sr

y, sr = load_audio(AUDIO_PATH)
dur = len(y) / sr
print(f"loaded {dur:.1f}s @ {sr} Hz  ({len(y):,} samples)")

## 3. Waveform, loudness, spectrograms

The **key plot** is the linear spectrogram zoomed to 0-300 Hz. The GX390 shows up as a ladder of evenly spaced harmonics; the bottom rung is the firing fundamental. Use it to set `FMIN/FMAX`.

In [ ]:
rms = librosa.feature.rms(y=y, frame_length=4096, hop_length=HOP)[0]
t_rms = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=HOP)

fig, ax = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
ax[0].plot(np.linspace(0, dur, len(y)), y, lw=0.3); ax[0].set_ylabel("amp"); ax[0].set_title("Waveform")
ax[1].plot(t_rms, librosa.amplitude_to_db(rms, ref=np.max), color="firebrick")
ax[1].set_ylabel("RMS (dB)"); ax[1].set_xlabel("time (s)"); ax[1].set_title("Loudness envelope")
plt.tight_layout(); plt.show()

In [ ]:
S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP))
S_db = librosa.amplitude_to_db(S, ref=np.max)

fig, ax = plt.subplots(figsize=(14, 5))
img = librosa.display.specshow(S_db, sr=sr, hop_length=HOP, x_axis="time", y_axis="linear",
                               ax=ax, cmap="magma")
ax.set_ylim(0, 300)
ax.set_title("STFT 0-300 Hz - the GX390 harmonic ladder; bottom rung = firing fundamental")
fig.colorbar(img, ax=ax, format="%+2.0f dB"); plt.tight_layout(); plt.show()

In [ ]:
M = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP, n_mels=128)
M_db = librosa.power_to_db(M, ref=np.max)
fig, ax = plt.subplots(figsize=(14, 5))
img = librosa.display.specshow(M_db, sr=sr, hop_length=HOP, x_axis="time", y_axis="mel",
                               ax=ax, cmap="magma")
ax.set_title("Mel spectrogram (full range - wind & road texture)")
fig.colorbar(img, ax=ax, format="%+2.0f dB"); plt.tight_layout(); plt.show()

## 4. Firing frequency -> RPM

Each candidate fundamental in `[FMIN, FMAX]` is scored by summing spectrum energy at `f0, 2f0 ... N_HARM·f0`. The exhaust pulse has a clean harmonic stack; broadband wind and road roar don't, so the engine wins the score **even if the AirPod thinned out the fundamental** and even when wind is louder in raw amplitude. A parabolic interpolation refines f0 below the FFT grid for a smoother trace.

In [ ]:
def track_engine_freq(S, sr, fmin, fmax, n_fft, n_harm):
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    df = sr / n_fft
    cand = np.where((freqs >= fmin) & (freqs <= fmax))[0]
    cf = freqs[cand]                                            # (C,)

    h = np.arange(1, n_harm + 1)
    hb = np.clip(np.round(np.outer(cf, h) / df).astype(int), 0, S.shape[0] - 1)  # (C,H)
    scores = S[hb, :].sum(axis=1)                              # (C,T)

    b = np.argmax(scores, axis=0)                              # (T,)
    C, T = scores.shape
    ti = np.arange(T)
    s0 = scores[b, ti]
    sm = scores[np.clip(b - 1, 0, C - 1), ti]
    sp = scores[np.clip(b + 1, 0, C - 1), ti]
    denom = sm - 2 * s0 + sp
    delta = np.where(denom != 0, 0.5 * (sm - sp) / denom, 0.0)
    delta = np.clip(delta, -0.5, 0.5)
    edge = (b == 0) | (b == C - 1)
    f0 = np.where(edge, cf[b], cf[b] + delta * df)
    return f0, s0

f0, conf = track_engine_freq(S, sr, FMIN, FMAX, N_FFT, N_HARM)
t_f0 = librosa.frames_to_time(np.arange(len(f0)), sr=sr, hop_length=HOP)

f0_s = medfilt(f0, 9)
conf_n = conf / np.max(conf)
f0_clean = np.where(conf_n > 0.15, f0_s, np.nan)
rpm = f0_clean * RPM_PER_HZ

valid = rpm[~np.isnan(rpm)]
if valid.size:
    print(f"RPM  min={np.nanmin(valid):.0f}  median={np.nanmedian(valid):.0f}  max={np.nanmax(valid):.0f}")
    print("GX390 can't exceed ~4500 rpm. If max is ~2x that, set HARMONIC_TRACKED=2 and re-run.")

### Harmonic ladder with the tracked fundamental

Solid line = tracked firing fundamental; dashed = its 2nd/3rd/4th harmonics. They should sit on bright rungs. If they're in the hash, tighten `FMIN/FMAX`.

In [ ]:
show = np.where(conf_n > 0.15, f0_s, np.nan)
fig, ax = plt.subplots(figsize=(14, 5))
img = librosa.display.specshow(S_db, sr=sr, hop_length=HOP, x_axis="time", y_axis="linear",
                               ax=ax, cmap="magma")
ax.plot(t_f0, show, color="lime", lw=1.6, label="firing fundamental")
for k in (2, 3, 4):
    ax.plot(t_f0, show * k, color="cyan", lw=0.8, ls="--", alpha=0.7)
ax.set_ylim(0, 250); ax.set_title("Tracked engine ladder"); ax.legend(loc="upper right")
fig.colorbar(img, ax=ax, format="%+2.0f dB"); plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t_f0, rpm, color="navy", lw=1.2)
ax.axhline(4500, color="0.6", ls=":", lw=1, label="~limiter")
ax.set_xlabel("time (s)"); ax.set_ylabel("estimated RPM")
ax.set_title("RPM trace - peaks = straights, dips = braking/corners (swing is narrow on a governed GX390)")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5. Load / effort proxies

On a quiet, governed engine these often read driving effort better than RPM. Loudness and **spectral centroid** both rise with load and speed-borne wind.

In [ ]:
cent = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=HOP)[0]
t_c = librosa.frames_to_time(np.arange(len(cent)), sr=sr, hop_length=HOP)
fig, ax = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
ax[0].plot(t_rms, uniform_filter1d(rms, 5), color="firebrick"); ax[0].set_ylabel("RMS"); ax[0].set_title("Loudness")
ax[1].plot(t_c, uniform_filter1d(cent, 5), color="darkgreen"); ax[1].set_ylabel("centroid (Hz)")
ax[1].set_xlabel("time (s)"); ax[1].set_title("Spectral brightness")
plt.tight_layout(); plt.show()

### Throttle on/off events

From sharp rises/falls in loudness. Tune `SENS` for more/fewer markers. (External-mic proxy - catches big transitions, not trail-brake nuance.)

In [ ]:
SENS = 2.0
env = uniform_filter1d(rms, 5)
d = np.gradient(env); thr = SENS * np.std(d)
on  = t_rms[np.where(d >  thr)[0]]
off = t_rms[np.where(d < -thr)[0]]
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t_rms, env, color="0.4")
ax.vlines(on, 0, env.max(), color="green", alpha=0.5, lw=0.8, label="on")
ax.vlines(off, 0, env.max(), color="red", alpha=0.5, lw=0.8, label="off")
ax.set_title("Throttle transitions"); ax.set_xlabel("time (s)"); ax.legend()
plt.tight_layout(); plt.show()
print(f"{len(on)} on, {len(off)} off events")

## 6. Lap-time estimate (cross-check your GPS dashboard)

A clean lap repeats its rhythm. Autocorrelating the combined RPM+loudness signal and finding the dominant period in a plausible lap window gives an audio-only lap time - compare it against the official splits from your Gateway Kartplex Sensor Logger pipeline.

In [ ]:
LAP_MIN, LAP_MAX = 20.0, 90.0   # seconds

# combine normalized rpm + loudness so it works even when RPM swing is small
r = np.nan_to_num(rpm); r = (r - r.mean()) / (r.std() + 1e-9)
L = np.interp(t_f0, t_rms, env); L = (L - L.mean()) / (L.std() + 1e-9)
sig = r + L

ac = np.correlate(sig, sig, mode="full")[len(sig) - 1:]; ac /= ac[0]
lags = np.arange(len(ac)) * HOP / sr
win = (lags >= LAP_MIN) & (lags <= LAP_MAX)
if win.any():
    lap = lags[win][np.argmax(ac[win])]
    print(f"estimated lap time ~ {lap:.1f}s")
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(lags, ac, color="purple"); ax.axvline(lap, color="orange", ls="--", label=f"{lap:.1f}s")
    ax.set_xlim(0, LAP_MAX * 1.5); ax.set_xlabel("lag (s)"); ax.set_ylabel("autocorr")
    ax.set_title("Autocorrelation - peak = lap period"); ax.legend()
    plt.tight_layout(); plt.show()
else:
    print("no lags in window; widen LAP_MIN/LAP_MAX")

## 7. Export datasets for sensor-fusion comparison

Writes everything to `out/` so you can join it against your Sensor Logger GPS/IMU export:
- `audio_timeseries.csv` - **detailed**, frame-level (~43 Hz): the master time series to resample & merge.
- `throttle_events.csv` - **detailed**: on/off transitions with slope.
- `lap_summary.csv` - **overview**: per-lap audio aggregates.
- `session_summary.json` - **overview**: config + headline conclusions + caveats.
- `comparison_prompt.md` - the cross-validation prompt (section 8).

All audio times are seconds from **recording start** - they are NOT yet aligned to the Sensor Logger clock. Alignment is the first job in the prompt.

In [ ]:
import pandas as pd, json, os
OUT_DIR = "out"; os.makedirs(OUT_DIR, exist_ok=True)

# put every feature on the f0 frame grid
rms_on_f0   = np.interp(t_f0, t_rms, rms)
rmsdb_on_f0 = np.interp(t_f0, t_rms, librosa.amplitude_to_db(rms, ref=np.max))
cent_on_f0  = np.interp(t_f0, t_c,  cent)
engine_on   = (conf_n > 0.15)

ts = pd.DataFrame({
    "t_audio_s":            np.round(t_f0, 4),
    "firing_hz":            np.round(f0_s, 3),
    "rpm":                  np.round(rpm, 1),
    "rpm_confidence":       np.round(conf_n, 3),
    "engine_on":            engine_on.astype(int),
    "rms":                  np.round(rms_on_f0, 6),
    "rms_db":               np.round(rmsdb_on_f0, 2),
    "spectral_centroid_hz": np.round(cent_on_f0, 1),
})
ts.to_csv(f"{OUT_DIR}/audio_timeseries.csv", index=False)
print("audio_timeseries.csv", ts.shape)

In [ ]:
# throttle events (recompute indices so we can attach slope magnitude)
env = uniform_filter1d(rms, 5)
d = np.gradient(env); thr = SENS * np.std(d)
on_idx  = np.where(d >  thr)[0]
off_idx = np.where(d < -thr)[0]
ev = pd.DataFrame({
    "t_audio_s": np.round(t_rms[np.concatenate([on_idx, off_idx])], 3),
    "event":     ["on"] * len(on_idx) + ["off"] * len(off_idx),
    "slope":     np.round(np.concatenate([d[on_idx], d[off_idx]]), 6),
}).sort_values("t_audio_s").reset_index(drop=True)
ev.to_csv(f"{OUT_DIR}/throttle_events.csv", index=False)
print("throttle_events.csv", ev.shape)

In [ ]:
# per-lap overview.
# BEST: paste authoritative S/F-crossing times (seconds from AUDIO start, after
# clock alignment) from your Sensor Logger lap pipeline. Otherwise we fall back
# to tiling the audio-estimated lap period - coarse, flagged as approximate.
GPS_LAP_BOUNDARIES_S = []   # e.g. [12.4, 51.8, 90.9, 130.1, ...]

if len(GPS_LAP_BOUNDARIES_S) >= 2:
    bounds = np.array(GPS_LAP_BOUNDARIES_S, float); lap_source = "gps"
else:
    period = lap if "lap" in globals() else dur
    start = t_f0[np.nanargmax(np.nan_to_num(rpm))] % period
    bounds = np.arange(start, dur, period); lap_source = "audio_estimate"

rows = []
for i in range(len(bounds) - 1):
    a, b = bounds[i], bounds[i + 1]
    m = (t_f0 >= a) & (t_f0 < b)
    if m.sum() == 0:
        continue
    seg = rpm[m]; has = np.any(~np.isnan(seg))
    rows.append({
        "lap": i + 1, "t_start_s": round(a, 2), "t_end_s": round(b, 2),
        "lap_time_s": round(b - a, 2),
        "rpm_mean": round(np.nanmean(seg), 0) if has else None,
        "rpm_max":  round(np.nanmax(seg), 0)  if has else None,
        "rms_mean": round(np.nanmean(rms_on_f0[m]), 5),
        "centroid_mean": round(np.nanmean(cent_on_f0[m]), 0),
        "engine_on_frac": round(np.mean(engine_on[m]), 2),
        "lap_source": lap_source,
    })
laps = pd.DataFrame(rows)
laps.to_csv(f"{OUT_DIR}/lap_summary.csv", index=False)
print(f"lap_summary.csv ({lap_source})"); print(laps)

In [ ]:
valid = rpm[~np.isnan(rpm)]
summary = {
    "source": {"audio_path": AUDIO_PATH, "duration_s": round(dur, 2),
               "sample_rate_hz": sr, "frame_rate_fps": round(sr / HOP, 2)},
    "config": {"engine": "Honda GX390 (4-stroke single, single-speed)",
               "FMIN": FMIN, "FMAX": FMAX, "HARMONIC_TRACKED": HARMONIC_TRACKED,
               "RPM_PER_HZ": RPM_PER_HZ, "N_FFT": N_FFT, "HOP": HOP, "N_HARM": N_HARM},
    "audio_conclusions": {
        "rpm_min":    float(np.nanmin(valid))    if valid.size else None,
        "rpm_median": float(np.nanmedian(valid)) if valid.size else None,
        "rpm_max":    float(np.nanmax(valid))    if valid.size else None,
        "engine_on_fraction": round(float(np.mean(engine_on)), 3),
        "throttle_on_events":  int(len(on_idx)),
        "throttle_off_events": int(len(off_idx)),
        "estimated_lap_time_s": round(float(lap), 2) if "lap" in globals() else None,
        "n_laps": int(len(laps)),
    },
    "caveats": [
        "RPM from exhaust harmonic; AirPod voice high-pass may thin the fundamental.",
        "engine_on==0 frames are wind/road dominated - treat as unreliable.",
        "GX390 is governed (<~4500 rpm); higher implies a tracked harmonic (raise HARMONIC_TRACKED).",
        "Single-speed kart: RPM should track GPS speed near-linearly - use that to validate & calibrate.",
    ],
}
with open(f"{OUT_DIR}/session_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## 8. Cross-validation prompt

Run the export cell below to drop `comparison_prompt.md` into `out/`. Hand that file, plus `audio_timeseries.csv` / `lap_summary.csv` / `session_summary.json` and your Sensor Logger GPS export, to an LLM (or me) to fuse the two pipelines.

In [ ]:
PROMPT = r"""# Prompt: cross-validate kart audio telemetry against Sensor Logger GPS

You are validating driving conclusions I extracted from an **in-helmet AirPods recording** of a Sodi GT5 rental kart (Honda GX390, 4-stroke single, **single-speed** clutch drive) against **Sensor Logger GPS/IMU telemetry** from the same session at Gateway Kartplex Track 1.

## Inputs
Audio-derived (from `out/`):
- `audio_timeseries.csv` - frame-level ~43 Hz. Cols: `t_audio_s` (s from recording start), `firing_hz`, `rpm`, `rpm_confidence` (0-1), `engine_on` (1=reliable), `rms`, `rms_db`, `spectral_centroid_hz`.
- `throttle_events.csv` - `t_audio_s`, `event` (on/off), `slope`.
- `lap_summary.csv` - per-lap audio aggregates (`lap_source` says whether boundaries are GPS-authoritative or audio-estimated/approximate).
- `session_summary.json` - config, headline conclusions, caveats.

Sensor Logger (I will attach): GPS Location (time as epoch ns or `seconds_elapsed`, latitude, longitude, speed m/s, course) plus my pipeline's lap numbers and S/F-crossing times; optionally Accelerometer/Gravity/Gyroscope.

## Step 1 - clock alignment (do this first)
The audio clock (t=0 at recording start) and the Sensor Logger clock are NOT synced. Find an offset `delta` mapping `t_audio_s -> t_gps`:
a. Resample audio `rms` (or `rpm`) and GPS `speed` to a common grid and cross-correlate; the lag at peak = `delta`.
b. If that's weak, align the audio-estimated lap period to the GPS lap period and match a distinctive feature (e.g. the slowest corner).
Report `delta`, the method, and the peak correlation. If r < ~0.5, say alignment is weak and mark all downstream results low-confidence.

## Step 2 - exploit the single-speed prior (the main validation)
No gearbox => engine RPM and ground speed are near-linear above clutch engagement. On reliable frames (`engine_on==1`):
- Regress audio `rpm` on GPS `speed`. Report slope, intercept, R^2.
- High R^2 validates the audio RPM extraction. If audio `rpm` is ~2x or ~3x the scale the regression implies, I tracked a harmonic - tell me the corrected `HARMONIC_TRACKED`.
- Flag low-speed departures from linearity as clutch slip / corner exit.

## Step 3 - comparisons
1. RPM vs speed correlation + regression (above) - headline.
2. Throttle-off events vs GPS deceleration: for each audio `off`, is there a speed drop / negative longitudinal accel within +/-1 s? Report hit rate and median timing error. Same for `on` vs corner-exit acceleration.
3. Lap times: compare `estimated_lap_time_s` and per-lap `lap_time_s` to my GPS S/F splits. Table: lap | GPS | audio | delta. Separate systematic bias from noise.
4. Per-lap pace: do audio `rpm_mean`/`rms_mean` rank laps the same way GPS lap time does? Spearman rho.
5. Corners: do audio RPM dips coincide with GPS low-speed points? Spot-check the slowest 3.

## Step 4 - diagnose disagreements (don't paper over them)
Attribute each mismatch to the most likely cause: wind/road noise (low `rpm_confidence`), AirPod fundamental dropout, clutch slip, GPS speed lag/error, or misalignment. Name the specific frames/laps.

## Output
- Applied `delta` + alignment confidence.
- RPM<->speed regression (slope, intercept, R^2) and the verdict on `HARMONIC_TRACKED`.
- Lap-time comparison table.
- Throttle-event <-> GPS-accel hit rates.
- Concrete notebook config changes for the next run (FMIN/FMAX/HARMONIC_TRACKED/confidence threshold).
Keep it terse and quantitative. Where the audio is low-confidence, say so plainly instead of over-claiming.
"""
with open(f"{OUT_DIR}/comparison_prompt.md", "w") as f:
    f.write(PROMPT)
print("wrote comparison_prompt.md", len(PROMPT), "chars")